# Splice — Adaptive Movie Recommender Engine

A movie recommender that switches strategy based on how much is known about a user — from cold-start
genre matching, to item-based KNN, to a stacked ensemble of five collaborative-filtering models blended
by a LightGBM meta-learner, optionally hybridized with content embeddings.

This notebook loads pre-trained models from `models/` and walks through how the system works and why
it was built this way. All the actual logic lives in `src/` — this notebook is a guided tour and a
record of the engineering decisions, not where the modelling happens.

Benchmarked on two datasets:

| Dataset | Users | Movies | Ratings | Test RMSE |
|---|---|---|---|---|
| MovieLens 100k | 943 | 1,682 | 100,000 | 0.9026 |
| MovieLens 1M | 6,040 | 3,706 | 1,000,209 | 0.8480 |

If you want to retrain from scratch, see the last section.

## Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src.*` works when running from notebooks/

import pickle
import pandas as pd

from src.data import load_and_clean
from src.engines import (
    get_popularity_recs,
    get_knn_recs,
    build_ensemble_predict,
    recommend,
    GENRE_PROFILES,
    get_profile_recs,
    score_by_genre_match,
)
from src.evaluate import evaluate_on_test, precision_recall_at_k
from src.embeddings import load_content_index, get_similar_by_plot
from src.hybrid import build_hybrid_predict

DATASET = "ml-100k"  # switch to "ml-1m" to explore the larger benchmark instead
MODELS_DIR = f"../models/{DATASET}"

## Load data and trained models

`load_and_clean()` re-reads and re-merges the raw source files — this part is cheap, so we just rerun
it rather than pickling the master tables. The actual trained models (which took real compute to
produce — 10-fold cross-validation across five algorithms) are loaded from disk.

In [ ]:
if DATASET == "ml-1m":
    from src.data_1m import load_and_clean as load_and_clean_1m
    data = load_and_clean_1m(data_dir="../data/ml-1m")
else:
    data = load_and_clean(data_dir="../data")

train_master = data["train_master"]
test_master = data["test_master"]
movie_info = data["movie_info"]
user_info = data["user_info"]
genre_cols = data["genre_cols"]

print(f"Train: {train_master.shape}, Test: {test_master.shape}")
print(f"Users: {user_info['user_id'].nunique()}, Movies: {movie_info['movie_id'].nunique()}")

In [ ]:
with open(f"{MODELS_DIR}/base_models.pkl", "rb") as f:
    base_models = pickle.load(f)

with open(f"{MODELS_DIR}/meta_learner.pkl", "rb") as f:
    meta = pickle.load(f)

with open(f"{MODELS_DIR}/popularity_lists.pkl", "rb") as f:
    popularity_lists = pickle.load(f)

with open(f"{MODELS_DIR}/knn.pkl", "rb") as f:
    knn = pickle.load(f)

with open(f"{MODELS_DIR}/stat_features.pkl", "rb") as f:
    stat_features = pickle.load(f)

print("Base models:", list(base_models.keys()))

In [ ]:
ensemble_predict = build_ensemble_predict(
    base_models,
    meta,
    stat_features["user_stats"],
    stat_features["item_stats"],
    stat_features["global_mean"],
)

## A note on SVD++

The original model set included SVD++, a variant of SVD that also factors in implicit feedback
(which items a user rated, regardless of the rating value). It's usually one of the stronger models
on MovieLens-style data, so it was included as a base learner going in.

On out-of-fold evaluation it came back with an RMSE of **1.0755** — worse than the plain `Baseline`
model (0.9464), and the weakest of everything tested. It was also the slowest model in the set by a
wide margin. Rather than let it drag the ensemble down while costing the most training time of any
model, it was dropped. The final base model set is SVD, item-based KNN, Baseline, SlopeOne, and
Co-Clustering.

## Base model performance (out-of-fold)

In [ ]:
with open(f"{MODELS_DIR}/results.txt") as f:
    print(f.read())

## A note on hyperparameter tuning

A random-search tuning pass (`src/tune.py`) was run over each base model and the meta-learner, using
a single train/validation split for speed rather than full 10-fold cross-validation per candidate.

- **SVD genuinely improved**: OOF RMSE went from 0.9725 to 0.9336 on 100k, and even more on 1M
  (0.9254 → 0.8710). This config was kept.
- **iKNN and Co-Clustering** landed within noise of their original settings — kept as-is rather than
  introducing single-split-search risk for no real gain.
- **The meta-learner's tuned config was tried and reverted.** It looked better on the single
  validation split used during search, but under the real 10-fold OOF procedure it regressed
  (0.8684 → 0.8937 on 100k) — a textbook case of overfitting to one split rather than generalizing.
  The original meta-learner hyperparameters were restored.

This is why a "tuning pass" isn't a single clean win-or-lose story here — some of it helped, some of
it didn't, and the honest result is a partial adoption rather than a wholesale swap.

## Cold start — the policy, not just the code

A collaborative-filtering ensemble is only as good as the rating history it has to work with. A
brand-new user has none, so scoring them through SVD, item-KNN, or the meta-learner isn't just
weaker — it's meaningless, since those models have no signal for a user_id they've never seen.

Splice routes on rating count rather than forcing every user through one model:

| Ratings on file | Engine used | Why |
|---|---|---|
| 0 | Genre-affinity scoring | No collaborative signal exists yet. |
| 1–4 | Item-based KNN | Enough for "movies similar to the one you rated highly" to work. |
| 5+ | Full stacked ensemble | Enough history for the meta-learner's blending to mean something. |

The dashboard's quiz and taste-profile browser are both cold-start by definition, and the UI states
this explicitly rather than presenting genre-matched picks as if they came from the trained ensemble.

In [ ]:
print("Top 10 overall (cold start, no history):")
for title in get_popularity_recs(popularity_lists, gender=None, n=10):
    print(" -", title)

## Sparse users — item-based KNN

In [ ]:
example_movie_id = movie_info.iloc[0]["movie_id"]
example_title = movie_info.iloc[0]["title"]
print(f"Because you liked \"{example_title}\":")
for title in get_knn_recs(example_movie_id, knn, movie_info, n=10):
    print(" -", title)

## Established users — the full ensemble

For users with 5+ ratings, every candidate movie gets scored by all five base models, and the
meta-learner combines those scores — plus per-user and per-item bias features — into a final
prediction.

In [ ]:
established_user_id = train_master.groupby("user_id").size().loc[lambda s: s >= 5].index[0]

recs = recommend(
    established_user_id, train_master, movie_info, popularity_lists, knn,
    ensemble_predict, user_info=user_info, n=10,
)

for movie_id, title, score in recs:
    print(f"  {score:.2f}  {title}")

## Hybridization — content embeddings + FAISS

The content-based signal comes from TMDB synopses, embedded with `sentence-transformers`
(`all-MiniLM-L6-v2`) and indexed with FAISS for cosine similarity search. This powers two things:

- **"Similar by plot"** — direct nearest-neighbor lookup, genuinely content-based, distinct from the
  collaborative KNN engine above (which finds movies rated similarly by the same users, regardless of
  what either movie is actually about).
- **Hybrid blend** — a per-user "taste vector" (average embedding of their highly-rated movies)
  blended with the ensemble's collaborative score at 85/15 weight.

This is a **post-hoc blend at inference time**, not a feature threaded into the meta-learner's
leakage-safe OOF training. Properly integrating content similarity into the stacking pipeline would
mean recomputing it per-fold with the same rigor as the statistical features — a larger rebuild than
was justified for what's ultimately a secondary signal here.

In [ ]:
content_index, content_movie_ids = load_content_index(MODELS_DIR if isinstance(MODELS_DIR, __import__("pathlib").Path) else __import__("pathlib").Path(MODELS_DIR))

if content_index is not None:
    similar = get_similar_by_plot(example_movie_id, content_index, content_movie_ids, movie_info, n=5)
    print(f"Similar by plot to \"{example_title}\":")
    for title in similar:
        print(" -", title)
else:
    print("Content index not built yet — run `python -m src.embeddings` first.")

In [ ]:
if content_index is not None:
    with open(f"{MODELS_DIR}/content_embeddings.pkl", "rb") as f:
        content_embeddings = pickle.load(f)

    hybrid_predict = build_hybrid_predict(
        ensemble_predict, train_master, content_movie_ids, content_embeddings
    )

    print("Pure ensemble:")
    for movie_id, title, score in recommend(
        established_user_id, train_master, movie_info, popularity_lists, knn,
        ensemble_predict, user_info=user_info, n=5,
    ):
        print(f"  {score:.2f}  {title}")

    print("\nHybrid (85% ensemble / 15% content):")
    for movie_id, title, score in recommend(
        established_user_id, train_master, movie_info, popularity_lists, knn,
        hybrid_predict, user_info=user_info, n=5,
    ):
        print(f"  {score:.2f}  {title}")

### A finding worth documenting: content-match quality

During development, "similar by plot" was found to sometimes return weak, coincidental matches once
genuinely similar movies ran out of the catalog — investigated by printing actual synopsis text and
cosine similarity scores side by side (`analysis/inspect_embeddings.py`) rather than guessing from
titles alone.

Root cause: on a catalog this size with short 1-3 sentence TMDB synopses, once the truly similar
movies for a query are exhausted, FAISS still returns *something* as the nearest neighbor — just not
a meaningfully similar one. Mitigated with a measured similarity threshold (0.47, calibrated from a
real query's score distribution) below which the feature reports "no strong matches" rather than
presenting a coincidental vocabulary overlap as a confident recommendation.

Notably, this improved substantially on the larger 1M catalog — Star Wars: A New Hope's actual trilogy
sequels scored 0.60+ with a sharp drop to under 0.40 for unrelated films, a much cleaner separation
than the 100k catalog could produce. This is exactly the expected behavior: catalog size was the
limiting factor, not a bug in the embedding or search logic. Full write-up was investigated and closed
as a GitHub issue.

## Evaluation

`evaluate_on_test` runs the held-out test set through the ensemble exactly once — this is the only
place `test_master` gets used for anything other than loading.

In [ ]:
test_rmse = evaluate_on_test(test_master, train_master, ensemble_predict)

### Ranking quality — precision/recall @10

RMSE tells you how close individual rating predictions are, but the actual product is a ranked list.
Precision@10 asks: of the top 10 recommended movies, how many did the user actually rate 4 or 5 stars?
Recall@10 asks: of all the movies this user rated highly, how many showed up in the top 10?

In [ ]:
pr = precision_recall_at_k(test_master, train_master, ensemble_predict, k=10)
print(f"Precision@10: {pr['precision_at_k']:.3f}")
print(f"Recall@10:    {pr['recall_at_k']:.3f}")
print(f"Evaluated across {pr['users_evaluated']} test users")

## Engineering decisions worth naming

A few judgment calls that shaped this project, kept honest rather than polished after the fact:

- **SVD++ was dropped** based on out-of-fold evidence, not assumption.
- **Testing caught two real bugs.** `tests/test_engines.py` revealed `get_knn_recs` could crash on a
  small catalog, and — more significantly — that the ensemble's candidate generation only ever
  considered movies present in `train_master`, meaning any movie with zero ratings from anyone could
  never be recommended regardless of predicted quality. Both fixed; RMSE was unaffected, confirming
  the bug was silent rather than accuracy-affecting.
- **Content-embedding matching quality was investigated, not assumed** — see the hybridization
  section above.
- **Hyperparameter tuning was a mixed result**, partially adopted rather than wholesale applied — see
  the tuning note above.
- **Two datasets, trained and saved independently** — 100k for fast iteration and the live dashboard
  demo, 1M as a "does this scale" benchmark. The two use different train/test split methodology
  (100k uses GroupLens's own split; 1M uses a random split), so the RMSE improvement moving to 1M
  isn't presented as a rigorously controlled scaling experiment — just a consistent, real result
  worth reporting honestly.

## Retraining from scratch

Not needed to explore this notebook — the cells above load pre-trained models. Only run this if
you've changed something in `src/` and want fresh results. The 10-fold cross-validation step takes
several minutes on 100k, proportionally longer on 1M.

In [ ]:
# from src.train import main
# main(dataset="ml-100k")

# To rebuild the content index (needed for hybridization):
# from src.embeddings import build_content_index
# build_content_index(movie_info, output_dir=__import__("pathlib").Path(MODELS_DIR))